# Кросс-энкодер без заголовков (EXP-026)

EXP-025: кросс-энкодер `BAAI/bge-reranker-v2-m3` отделяет «свой» слайд тезиса от «чужого» намного лучше косинуса
и точнее размечает настоящие ответы GRPO, но в парах «тезис не раскрыт» наказывает пропуск не сильнее принятой
награды. Разбивка по видам пар: сильнее принятой там, где у слайда другое содержимое (`drop_explanation`,
`foreign_body`, `overloaded_slide`), слабее там, где остаётся заголовок тезиса без содержимого (`title_only_thesis`
−0,036 против −0,069, `empty_body`, `hide_in_*`). Гипотеза: вероятность держится на совпадении заголовков — запрос
начинался с заголовка тезиса, документ — с заголовка слайда.

Здесь запрос — только утверждение тезиса, документ — содержимое слайда без заголовка (`reranker_body`; слайд без
содержимого — 0): `rrb` — вместо косинуса, `rrb_share` — доля тезисов с вероятностью ≥ 0,5. Для сравнения — принятая
награда и `rr` (EXP-025). Пары, мерки и правило — как в EXP-025 (`decisions.md`, раздел 7, решение 16.11.2026,
EXP-026, записано до прогона). Результат — `runs/perturbation/<дата>/`.

In [ ]:
import csv, glob, json, os, statistics, sys, time
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # перед запуском проверить nvidia-smi: карта 0 свободна
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl import perturb_eval as pe
from presentation_rl.rewards import Lemmatizer, load_key_points
from presentation_rl.tasks import load_tasks

OUT = ROOT / "runs" / "perturbation" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "runs" / "perturbation" / "embeddings"  # общий кеш эмбеддингов и оценок кросс-энкодера
EMBEDDER, ALPHA, RERANKER = "BAAI/bge-m3", 0.2, "BAAI/bge-reranker-v2-m3"
CURRENT = pe.Config(EMBEDDER, ALPHA, *pe.CURRENT_WEIGHTS, "accepted")
RR = pe.Config(EMBEDDER, ALPHA, *pe.CURRENT_WEIGHTS, "rr")  # EXP-025, для сравнения, не выбирается
RRB = [pe.Config(EMBEDDER, ALPHA, *pe.CURRENT_WEIGHTS, v) for v in pe.BODY_VARIANTS]
CONFIGS = [CURRENT, RR, *RRB]
RUNS = [ROOT / "runs" / "variance" / "20261020_0331", ROOT / "runs" / "reward_sources" / "20261027_2150"]  # σ
MISSING = {"drop_explanation", "empty_body", "title_only_thesis", "foreign_body", "keys_only", "hide_in_notes",
           "hide_in_plan", "hide_in_image", "overloaded_slide", "generic_body"}  # тезис не раскрыт
INVARIANCE = {"paraphrase", "reorder"}
print(OUT)

In [ ]:
def write_csv(path, rows):
    rows = list(rows)
    if not rows:
        return
    fields = list(dict.fromkeys(k for r in rows for k in r))
    with open(path, "w", newline="", encoding="utf-8-sig") as f:  # utf-8-sig — для Excel
        writer = csv.DictWriter(f, fields)
        writer.writeheader()
        writer.writerows(rows)


points = load_key_points(ROOT / "data" / "tasks" / "key_points" / "key_points.json")
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json")}
SETS = {}
for name, folder in (("old", ROOT / "data" / "perturbation"), ("holdout", ROOT / "data" / "perturbation" / "holdout")):
    items, pairs = pe.load_variants(folder / "variants")
    annotations = json.loads((folder / "annotations.json").read_text(encoding="utf-8"))
    generic, generic_pairs = pe.generic_variants(items, annotations)
    items.update(generic)
    SETS[name] = {"items": items, "pairs": pairs + generic_pairs, "annotations": annotations}
    print(f"{name}: вариантов {len(items)}, пар {len(pairs)} + общие слова {len(generic_pairs)}, исходников {len(annotations)}")
contexts = {key[0]: item.context for key, item in SETS["old"]["items"].items() if key[1] == "original"}
natural = pe.load_natural(RUNS, tasks, set(contexts))
SETS["old"]["items"].update(natural)
conflict_items, conflicts = pe.load_conflicts(ROOT / "data" / "perturbation" / "conflicts", contexts)
SETS["old"]["items"].update(conflict_items)
pe.combine_judges(conflicts, "author", "codex", ["repeat_vs_missing"], "judge")  # судья — как в EXP-016/018
print("ответов T-lite для σ:", len(natural), "| пар с конфликтом:", len(conflicts))
start = time.time()
for name, s in SETS.items():
    pe.build_all(s["items"])
    s["pairs"], dropped = pe.check_pairs(s["pairs"], s["items"])
    print(f"{name}: после предусловий пар {len(s['pairs'])}, отброшено {len(dropped)}")
print(f"сборка {time.time() - start:.0f} с")

## Разделяет ли кросс-энкодер «свой» и «чужой» слайд тезиса

Размеченные слайды тезисов: «свой» — вероятность для своего слайда, «чужой» — лучшая для остальных слайдов той же
презентации; `rr` (EXP-025) и `rrb`. Шкала косинуса EXP-024 нужна только функции разметки.

In [ ]:
embedder = pe.CachedEmbedder(EMBEDDER, CACHE)
reranker = pe.CachedReranker(RERANKER, CACHE, batch_size=32)
lemmatizer = Lemmatizer()
SCALE, _ = pe.calibrate_coverage(SETS["old"]["items"], SETS["old"]["annotations"], points, embedder, lemmatizer,
                                 CURRENT.weights())
print(f"шкала косинуса (EXP-024): lo = {SCALE[0]:.4f}, hi = {SCALE[1]:.4f}")
separation = []
for config in (RR, RRB[0]):
    for name in SETS:
        start = time.time()
        _, rows = pe.calibrate_coverage(SETS[name]["items"], SETS[name]["annotations"], points, embedder, lemmatizer,
                                        config.weights(), reranker=reranker)
        reranker.save()
        own, other = [r["own"] for r in rows], [r["other"] for r in rows]
        auc = statistics.mean(float(o > t) for o in own for t in other)
        print(f"  {config.variant:4s} {name:8s}: тезисов {len(rows)}; свой {statistics.median(own):.3f}, "
              f"чужой {statistics.median(other):.3f}; свой > чужой {auc:.2f}; свой ≥ 0,5: "
              f"{statistics.mean(o >= 0.5 for o in own):.2f}; чужой < 0,5: {statistics.mean(t < 0.5 for t in other):.2f} "
              f"({time.time() - start:.0f} с)")
        separation += [{"config": config.variant, "set": name, **r} for r in rows]
write_csv(OUT / "reranker_separation.csv", separation)

In [ ]:
scores, sigmas, hold_scores = {}, {}, {}
for config in CONFIGS:
    start = time.time()
    scores[config] = pe.score_items(SETS["old"]["items"], points, config, embedder, lemmatizer, scale=SCALE,
                                    reranker=reranker)
    sigmas[config], _ = pe.sigma(scores[config])
    hold_scores[config] = pe.score_items(SETS["holdout"]["items"], points, config, embedder, lemmatizer, scale=SCALE,
                                         reranker=reranker)
    embedder.save()
    reranker.save()
    print(f"{config.variant:15s} σ = {sigmas[config]:.4f}  ({time.time() - start:.0f} с)")
write_csv(OUT / "components.csv", ({"config": c.variant, "set": name, "key": "/".join(map(str, k)), **v}
                                   for name, sc in (("old", scores), ("holdout", hold_scores))
                                   for c, s in sc.items() for k, v in s.items()))

## Сдвиги и выбор (правило — до прогона)

«Тезис не раскрыт» — медиана Δ в σ **своей** конфигурации (для GRPO важно отношение сигнала к разбросу). Перефразировка,
перестановка, `keys_only` и остальные семейства — с порогами от σ **принятой** награды (одна мерка для всех вариантов:
в EXP-024 доли «ключей» и «инструментов» падали только оттого, что порог рос вместе с σ).

Вариант допустим, если на основном наборе **и** на новых исходниках: (а) «тезис не раскрыт» ниже, чем у принятой,
хотя бы на 0,25σ; (б) перефразировка и перестановка — не ниже принятой минус 0,10; (в) `keys_only` — не реже
принятой; (г) остальные семейства (кроме покрытия и инвариантности) — не ниже принятой минус 0,05; и (д) совпадение
с разметкой пар с конфликтом не ниже принятой минус 0,05. Из допустимых — с наибольшим усилением на новых исходниках;
при разнице меньше 0,1σ — `rrb` (проще). Нет допустимых — покрытие прежнее. Ниже — разбивка «тезис не раскрыт» по
видам пар (Δ награды, медиана): проверка гипотезы о заголовках, не для выбора.

In [ ]:
def rows_for(name, config, sigma_value):
    s = scores[config] if name == "old" else hold_scores[config]
    rows = pe.evaluate_pairs(SETS[name]["pairs"], s, pe.M_SHARE * sigma_value, pe.EPS_SHARE * sigma_value)
    for r in rows:
        r["d_sigma"] = r["d_total"] / sigmas[config]
    return rows


OWN = {(n, c): rows_for(n, c, sigmas[c]) for n in SETS for c in CONFIGS}  # пороги от своей σ
BASE = {(n, c): rows_for(n, c, sigmas[CURRENT]) for n in SETS for c in CONFIGS}  # пороги от σ принятой
write_csv(OUT / "pairs.csv", ({"set": n, "config": c.variant, **r} for (n, c), rows in OWN.items() for r in rows))
AGREE = {c: pe.conflict_rows(conflicts, scores[c], "judge", pe.EPS_SHARE * sigmas[c]) for c in CONFIGS}


def metrics(name, config):
    own, base = OWN[(name, config)], BASE[(name, config)]
    missing = [r["d_sigma"] for r in own if r["kind"] in MISSING]
    mean = lambda v: statistics.mean(v) if v else None
    return {"missing": statistics.median(missing) if missing else None,
            "generic": statistics.median([r["d_sigma"] for r in own if r["kind"] == "generic_body"] or [0.0]),
            "invariance": mean([r["total"] == "ok" for r in base if r["kind"] in INVARIANCE and r["total"] is not None]),
            "keys_only": mean([r["total"] in ("ok", "weak") for r in base if r["kind"] == "keys_only"]),
            "families": pe.family_shares(pe.tally(base), {t: 1 for t in {r["task"] for r in base}}),
            "agree": mean([r["agree"] for r in AGREE[config]]) if name == "old" else None}


METRICS = {(n, c): metrics(n, c) for n in SETS for c in CONFIGS}
fmt = lambda v: "—" if v is None else f"{v:+.2f}" if v < 0 else f"{v:.2f}"
for name in SETS:
    print(f"== {name}: «тезис не раскрыт» (σ своей) | из них общие слова | перефразировка+перестановка | keys_only | "
          f"пары с конфликтом")
    for c in CONFIGS:
        m = METRICS[(name, c)]
        print(f"  {c.variant:15s} σ={sigmas[c]:.3f}  {fmt(m['missing']):>6s} | {fmt(m['generic']):>6s} | "
              f"{fmt(m['invariance']):>5s} | {fmt(m['keys_only']):>5s} | {fmt(m['agree'])}")
    print("  семейства (пороги от σ принятой):")
    for c in CONFIGS:
        print(f"    {c.variant:15s}", ", ".join(f"{f} {v:.2f}" for f, v in sorted(METRICS[(name, c)]["families"].items())))
summary = [{"set": n, "config": c.variant, "sigma": sigmas[c], **e} for (n, c), rows in OWN.items() for e in pe.summarize(rows)]
write_csv(OUT / "summary.csv", summary)

In [ ]:
for name in SETS:
    print(f"== {name}: Δ награды (медиана) по видам «тезис не раскрыт»; число пар")
    print(f"  {'вид':18s}" + "".join(f"{c.variant:>11s}" for c in CONFIGS))
    for kind in sorted(MISSING):
        cells = []
        for c in CONFIGS:
            d = [r["d_total"] for r in OWN[(name, c)] if r["kind"] == kind and r["d_total"] is not None]
            cells.append(f"{statistics.median(d):+11.3f}" if d else f"{'—':>11s}")
        n = sum(1 for r in OWN[(name, CURRENT)] if r["kind"] == kind)
        print(f"  {kind:18s}" + "".join(cells) + f"   {n}")

In [ ]:
report = []
for c in RRB:
    checks = {}
    for name in SETS:
        m, b = METRICS[(name, c)], METRICS[(name, CURRENT)]
        checks[name] = {
            "missing": m["missing"] <= b["missing"] - 0.25,
            "invariance": m["invariance"] is None or m["invariance"] >= b["invariance"] - 0.10,
            "keys_only": m["keys_only"] is None or b["keys_only"] is None or m["keys_only"] >= b["keys_only"],
            "families": all(m["families"].get(f, 0) >= v - 0.05 for f, v in b["families"].items()
                            if f not in ("coverage", "invariance"))}
    checks["old"]["agree"] = METRICS[("old", c)]["agree"] >= METRICS[("old", CURRENT)]["agree"] - 0.05
    ok = all(all(v.values()) for v in checks.values())
    gain = METRICS[("holdout", CURRENT)]["missing"] - METRICS[("holdout", c)]["missing"]
    report.append({"variant": c.variant, "admissible": ok, "gain_holdout": gain,
                   **{f"{n}_{k}": v for n, part in checks.items() for k, v in part.items()}})
    print(c.variant, "допустим" if ok else "нет", f"усиление на новых {gain:+.2f}σ", checks)
order = {v: i for i, v in enumerate(pe.BODY_VARIANTS)}
admissible = sorted((r for r in report if r["admissible"]), key=lambda r: -r["gain_holdout"])
CHOSEN = None
if admissible:
    best = admissible[0]["gain_holdout"]
    CHOSEN = min((r for r in admissible if best - r["gain_holdout"] < 0.1), key=lambda r: order[r["variant"]])["variant"]
write_csv(OUT / "choice.csv", report)
(OUT / "choice.json").write_text(json.dumps({"coverage": CHOSEN, "reranker": RERANKER}, ensure_ascii=False, indent=1),
                                 encoding="utf-8")
print("ИТОГ: покрытие —", CHOSEN or "прежнее")

## Сверка с ручным разбором: какие тезисы раскрыты в ответах GRPO

Ответы исходной модели и шага 130 EXP-023 на 8 темах validation (не для выбора). «+» — тезис раскрыт (вероятность
лучшего слайда ≥ 0,5), «−» — нет. Сверка по тексту (EXP-025): у шага 130 `plain_t376_1` раскрыты все пять тезисов
(«Фотография 51», шаг спирали 3,4 нм, слайд «Значение открытия»); у `search_t376_1` нет параметров спирали, модель
Уотсона и Крика — одной фразой; t389 — омбудсмен не раскрыт ни у кого. Каждой конфигурации — доля раскрытых.

In [ ]:
from presentation_rl.perturb import task_context
from presentation_rl.rewards import score as reward_score

RUN = sorted((ROOT / "runs" / "grpo").glob("2*_seed0"))[-1]
FOLDERS = {"исходная": ROOT / "runs" / "grpo" / "reference" / "episodes", "шаг 130": RUN / "checkpoints" / "step_0130" / "validation"}
topics = json.loads((RUN / "manifest.json").read_text(encoding="utf-8"))["validation_topics"]
grpo_items = {}
for label, folder in FOLDERS.items():
    for meta in sorted(folder.glob("*_*_*.json")):
        r = json.loads(meta.read_text(encoding="utf-8"))
        if r["task_id"] not in topics:
            continue
        stem = meta.stem
        key = (r["task_id"], label, stem)
        grpo_items[key] = pe.Item(key, r["task_id"], (folder / f"{stem}_answer.txt").read_text(encoding="utf-8"),
                                  task_context(tasks[r["task_id"]], (folder / f"{stem}.txt").read_text(encoding="utf-8"),
                                               r["mode"] == "search"))
pe.build_all(grpo_items)
marks_rows = []
for c in [RR, *RRB]:
    w = c.weights(scale=SCALE)
    print("==", c.variant)
    for key, item in sorted(grpo_items.items()):
        ctx = item.context
        r = reward_score(item.deck, points[item.task], conclusion=ctx["conclusion"],
                         slide_range=tuple(ctx["slide_range"]) if ctx["slide_range"] else None, embedder=embedder,
                         lemmatizer=lemmatizer, report=item.report, images_allowed=ctx["images_allowed"],
                         tool_text=ctx.get("tool_text"), calls=ctx.get("calls"), search=ctx["search"], weights=w,
                         reranker=reranker)
        covered = [t.best_slide is not None and t.similarity >= 0.5 for t in r.theses]
        marks = "".join("+" if x else "−" for x in covered)
        missing = [p.title[:32] for p, x in zip(points[item.task], covered) if not x]
        marks_rows.append({"config": c.variant, "task": item.task, "model": key[1], "episode": key[2], "marks": marks,
                           "coverage": r.coverage, "missing": "; ".join(missing)})
        if item.task in ("t376", "t389"):
            print(f"  {key[1]:9s} {key[2]:16s} {marks} | {'; '.join(missing)[:150]}")
    for label in FOLDERS:
        part = [m for m in marks_rows if m["config"] == c.variant and m["model"] == label]
        share = statistics.mean(m["marks"].count("+") / len(m["marks"]) for m in part)
        print(f"  {label}: доля раскрытых тезисов на 8 темах {share:.2f}")
reranker.save()
write_csv(OUT / "grpo_marks.csv", marks_rows)